# EDA — escenario tabular (crédito)

Exploración del conjunto UCI *Default of Credit Card Clients*.

Este notebook **no contiene lógica crítica**: solo llama a `uq.data.credit` y a `uq.eda`, que son las mismas funciones que usa `scripts/run_eda.py`. Así el informe y la tesis se regeneran con un comando y el notebook sirve para entender los datos.

Antes de ejecutarlo, descarga los datos:

```bash
uv run python scripts/download_data.py
```

In [ ]:
from IPython.display import Image, display

from uq.data import credit
from uq.eda import figures, summary
from uq.settings import figures_dir

frame = credit.load()
frame.shape

In [ ]:
# Columnas, tipos y primeras filas.
frame.dtypes.to_frame("dtype").T

In [ ]:
frame.head()

In [ ]:
# Resumen reproducible (el mismo que se guarda en results/eda/credit.json).
resumen = summary.tabular_summary(
    frame,
    credit.TARGET,
    sensitive=credit.SENSITIVE_COLUMNS[:3],
    categorical=credit.CATEGORICAL_COLUMNS,
)

print("Filas:", resumen["n_rows"])
print("Variables:", resumen["n_features"])
print("Faltantes:", resumen["n_missing_values"])
print("Duplicados (sin etiqueta):", resumen["n_duplicate_rows"])
print("Tasa de impago:", round(resumen["target"]["positive_rate"], 4))

In [ ]:
# Los duplicados hay que quitarlos ANTES de partir los datos: si no, la misma
# fila puede caer en entrenamiento y en prueba (fuga de información).
duplicados = frame[frame.drop(columns=[credit.TARGET]).duplicated(keep=False)]
duplicados.sort_values(["limit_bal", "age"]).head(10)

In [ ]:
# Tasa de impago por grupo demográfico (sesgo potencial que hay que documentar).
frame.groupby("sex", observed=True)[credit.TARGET].agg(["size", "mean"]).rename(
    columns={"size": "n", "mean": "tasa_impago"}
)

In [ ]:
# Figuras ya generadas por scripts/run_eda.py (una sola fuente de verdad).
figures.plot_credit_class_balance(frame, credit.TARGET, figures_dir() / "credit_class_balance.png")
figures.plot_credit_distributions(frame, figures_dir() / "credit_distributions.png")

display(Image(filename=str(figures_dir() / "credit_class_balance.png")))
display(Image(filename=str(figures_dir() / "credit_distributions.png")))